# Option IV Surface training - Kaggle Edition

This notebook is optimized for the **Kaggle "Save Version" (Submit and Wait)** pattern.
- Uses GPU T4 x2 or P100.
- Redirects all model outputs to `/kaggle/working/` for persistence.
- Handles private data if mounted via Kaggle Datasets.

## 1. Setup & Repository Cloning

In [ ]:
import os
import subprocess

# Clone the project
if not os.path.exists('option'):
    !git clone https://github.com/Uwater1/option

%cd option

## 2. Dependency Installation
Installing only the necessary packages to keep the environment clean.

In [ ]:
!pip install -r requirements.txt
# Ensure the latest versions for GPU support
!pip install -U xgboost lightgbm catboost optuna

## 3. Data Configuration
Kaggle datasets are usually stored in `/kaggle/input/`. 
Adjust the `DATA_DIR` if you have uploaded your `options_data` as a Kaggle Dataset.

In [ ]:
DATA_DIR = "options_data" # Default local path in repo

# If using a Kaggle Dataset, uncomment and update the path below:
# DATA_DIR = "/kaggle/input/your-dataset-name/options_data/"

if not os.path.exists(DATA_DIR):
    print(f"Warning: {DATA_DIR} not found. Ensure data is uploaded or path is correct.")

## 4. Run Training (Kaggle Wait Pattern)
We run the training script. All models produced will be moved to the output directory.
Note: `ydf` is automatically skipped if GPU is detected (configured in `train_prod_model.py`).

In [ ]:
!python train_prod_model.py --data-dir {DATA_DIR} --models all --tune --tune-trials 30

## 5. Persistence (Model Export)
Kaggle only saves files in `/kaggle/working/`. We move the models from our repo folder to the root output folder.

In [ ]:
!cp iv_prod_* /kaggle/working/ 2>/dev/null || : 
!cp -r iv_prod_ydf /kaggle/working/ 2>/dev/null || : 

print("Training complete. Models and optimized parameters (iv_prod_params.json) are now in /kaggle/working/ and will be available as Output artifacts.")

# Create an empty submission file if this is used in a competition context
with open('/kaggle/working/submission.csv', 'w') as f:
    f.write('id,prediction\n')
    f.write('0,0\n')